In [ ]:
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine/wine.data'

### Multiclass Classification
: 결과가 여러개의 클래스로 나오는 경우

## Wine Recognition Dataset
: 와인의 13가지 특징으로 3개의 품종중 어디에 속하는지 다중분류

- 데이터 개수 : 178개
- 입력 특징 : 13개
- 정답 클래스 : 3개(1,2,3 중 하나)

In [1]:

import os
import seaborn as sns
import torch
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from torch import nn, optim
import numpy as np
import pandas as pd

torch.manual_seed(42)

In [3]:
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine/wine.data'

columns = ['class',
            'Alcohol',
            'Malicacid',
            'Ash',
            'Alcalinity_of_ash',
            'Magnesium',
            'Total_phenols',
            'Flavanoids',
            'Nonflavanoid_phenols',
            'Proanthocyanins',
            'Color_intensity',
            'Hue',
            '0D280_0D315_of_diluted_wines',
            'Proline']

df = pd.read_csv(url, names=columns)
# print(df.head())
# print('데이터 크기 :', df.shape)
# print('클래스별 데이터 개수 :', df["class"].value_counts().sort_index())

# 입력(X) , 출력(y) 분리 및 원본 클래스 1,2,3 -> 0,1,2 변환
x = columns[1:]
X = torch.tensor(df[x].values, dtype=torch.float32)

y = df['class'].values-1
Y = torch.tensor(y, dtype=torch.long)

# train(80%), test(20%)로 분할
train_indices = []
test_indices = []

for class_id in [0, 1, 2]:
    class_indices = np.where(Y == class_id)[0]
    class_indices = class_indices[torch.randperm(len(class_indices)).numpy()]

    split = int(len(class_indices) * 0.8)
    train_indices.extend(class_indices[:split].tolist())
    test_indices.extend(class_indices[split:].tolist())

# 학습/테스트 인덱스 섞기
train_indices = torch.tensor(train_indices, dtype=torch.long)
test_indices = torch.tensor(test_indices, dtype=torch.long)

train_indices = train_indices[torch.randperm(len(train_indices))]
test_indices = test_indices[torch.randperm(len(test_indices))]

# 원본데이터 분리 (train입력, train출력, test 입력, test 출력)
X_train = X[train_indices.numpy()]
X_test = X[test_indices.numpy()]
Y_train = Y[train_indices.numpy()]
Y_test = Y[test_indices.numpy()]

# 표준화
mean = X_train.mean(dim=0)
std = X_train.std(dim=0).clamp(min=1e-5)

X_train_scaled = (X_train - mean) / std
X_test_scaled = (X_test - mean) / std

Y_train = torch.tensor(Y_train, dtype=torch.long)
Y_test = torch.tensor(Y_test, dtype=torch.long)

# 오분류 사례를 원래 단위로 확인하기 위한 테스트 데이터 보관
test_df=df.iloc[test_indices.numpy()]
print(test_df)

print('X_train :', X_train.shape, X_train.dtype)
print('Y_train :', Y_train.shape, Y_train.dtype)
print('X_test :', X_test.shape, X_test.dtype)
print('Y_test :', Y_test.shape, Y_test.dtype)

print('학습 데이터 클래스 개수 :', torch.bincount(Y_train, minlength=3))
print('테스트 데이터 클래스 개수 :', torch.bincount(Y_test, minlength=3))

     class  Alcohol  Malicacid   Ash  Alcalinity_of_ash  Magnesium  \
5        1    14.20       1.76  2.45               15.2        112   
144      3    12.25       3.88  2.20               18.5        112   
63       2    12.37       1.13  2.16               19.0         87   
166      3    13.45       3.70  2.60               23.0        111   
7        1    14.06       2.15  2.61               17.6        121   
152      3    13.11       1.90  2.75               25.5        116   
0        1    14.23       1.71  2.43               15.6        127   
141      3    13.36       2.56  2.35               20.0         89   
98       2    12.37       1.07  2.10               18.5         88   
4        1    13.24       2.59  2.87               21.0        118   
35       1    13.48       1.81  2.41               20.5        100   
40       1    13.56       1.71  2.31               16.2        117   
38       1    13.07       1.50  2.10               15.5         98   
116      2    11.82 

C:\Users\mega\AppData\Local\Temp\ipykernel_10360\2630121984.py:62: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  Y_train = torch.tensor(Y_train, dtype=torch.long)
C:\Users\mega\AppData\Local\Temp\ipykernel_10360\2630121984.py:63: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  Y_test = torch.tensor(Y_test, dtype=torch.long)
